In [18]:
import numpy as np
import pandas as pd

# -----------------------------
# Load training dataset
# -----------------------------

train_data = pd.read_csv("emnist-letters-train.csv")

# shuffle dataset
train_data = train_data.sample(frac=1).reset_index(drop=True)

# reduce size for faster debugging
#train_data = train_data.iloc[:10000]

X_train = train_data.iloc[:,1:].values / 255.0
labels_train = train_data.iloc[:,0].values

X_train = X_train.T

m_train = X_train.shape[1]


# -----------------------------
# Load test dataset
# -----------------------------

test_data = pd.read_csv("emnist-letters-test.csv")

X_test = test_data.iloc[:,1:].values / 255.0
labels_test = test_data.iloc[:,0].values

X_test = X_test.T


# -----------------------------
# One-hot encoding for training labels
# -----------------------------

Y_train = np.zeros((26, m_train))

for i,l in enumerate(labels_train):
    Y_train[l-1, i] = 1


# -----------------------------
# Initialize parameters
# -----------------------------

np.random.seed(1)

W1 = np.random.randn(64,784) * np.sqrt(2/784)
W2 = np.random.randn(32,64) * np.sqrt(2/64)
W3 = np.random.randn(26,32) * np.sqrt(2/32)

b1 = np.zeros((64,1))
b2 = np.zeros((32,1))
b3 = np.zeros((26,1))


# -----------------------------
# Activation functions
# -----------------------------

def sigmoid(z):
    return 1/(1+np.exp(-z))

def sigmoid_derivative(a):
    return a*(1-a)

def softmax(z):
    exp_z = np.exp(z - np.max(z,axis=0,keepdims=True))
    return exp_z / np.sum(exp_z,axis=0,keepdims=True)
def relu(z):
    return np.maximum(0,z)

def relu_derivative(z):
    return (z > 0).astype(float)

# -----------------------------
# Cost function
# -----------------------------

def compute_cost(A3,Y):
    
    m = Y.shape[1]
    
    cost = -(1/m)*np.sum(Y*np.log(A3 + 1e-8))
    
    return cost


# -----------------------------
# Forward propagation
# -----------------------------

def forward_prop(X):

    Z1 = W1.dot(X) + b1
    A1 = relu(Z1)

    Z2 = W2.dot(A1) + b2
    A2 = relu(Z2)

    Z3 = W3.dot(A2) + b3
    A3 = softmax(Z3)

    cache = (Z1,A1,Z2,A2,Z3,A3)

    return A3,cache


# -----------------------------
# Backpropagation
# -----------------------------

def backprop(X,Y,cache):

    global W1,W2,W3,b1,b2,b3

    Z1,A1,Z2,A2,Z3,A3 = cache

    m = X.shape[1]

    dZ3 = A3 - Y
    dW3 = (1/m)*dZ3.dot(A2.T)
    db3 = (1/m)*np.sum(dZ3,axis=1,keepdims=True)

    dZ2 = (W3.T.dot(dZ3))*relu_derivative(A2)
    dW2 = (1/m)*dZ2.dot(A1.T)
    db2 = (1/m)*np.sum(dZ2,axis=1,keepdims=True)

    dZ1 = (W2.T.dot(dZ2))*relu_derivative(A1)
    dW1 = (1/m)*dZ1.dot(X.T)
    db1 = (1/m)*np.sum(dZ1,axis=1,keepdims=True)

    return dW1,dW2,dW3,db1,db2,db3


# -----------------------------
# Update parameters
# -----------------------------

def update_params(dW1,dW2,dW3,db1,db2,db3,lr):

    global W1,W2,W3,b1,b2,b3

    W1 = W1 - lr*dW1
    W2 = W2 - lr*dW2
    W3 = W3 - lr*dW3

    b1 = b1 - lr*db1
    b2 = b2 - lr*db2
    b3 = b3 - lr*db3


# -----------------------------
# Prediction
# -----------------------------

def predict(X):

    A3,_ = forward_prop(X)

    predictions = np.argmax(A3,axis=0)

    return predictions


# -----------------------------
# Accuracy function
# -----------------------------

def accuracy(X,labels):

    preds = predict(X)

    labels = labels - 1

    acc = np.mean(preds == labels)

    return acc


# -----------------------------
# Training
# -----------------------------

learning_rate = 0.001
epochs = 6000
batch_size = 128

for epoch in range(epochs):
    learning_rate = 0.001 * (0.95 ** (epoch//500))
    for i in range(0, m_train, batch_size):

        X_batch = X_train[:, i:i+batch_size]
        Y_batch = Y_train[:, i:i+batch_size]

        A3, cache = forward_prop(X_batch)

        cost = compute_cost(A3, Y_batch)

        dW1,dW2,dW3,db1,db2,db3 = backprop(X_batch, Y_batch, cache)

        update_params(dW1,dW2,dW3,db1,db2,db3,learning_rate)

    if epoch % 100 == 0:
        print("Epoch:",epoch,"Cost:",cost)
        train_acc = accuracy(X_train,labels_train)
        test_acc = accuracy(X_test,labels_test)

        print("Training Accuracy:",train_acc)
        print("Test Accuracy:",test_acc)


# -----------------------------
# Evaluate model
# -----------------------------

train_acc = accuracy(X_train,labels_train)
test_acc = accuracy(X_test,labels_test)

print("Training Accuracy:",train_acc)
print("Test Accuracy:",test_acc)

Epoch: 0 Cost: 3.192799929178589
Training Accuracy: 0.07423507021475467
Test Accuracy: 0.07054530711534564
Epoch: 100 Cost: 1.0299967526277716
Training Accuracy: 0.7789839975675402
Test Accuracy: 0.7578890465571998
Epoch: 200 Cost: 0.783445955878361
Training Accuracy: 0.8333990247637924
Test Accuracy: 0.8049192513007636
Epoch: 300 Cost: 0.6375429454356988
Training Accuracy: 0.8567776664151623
Test Accuracy: 0.8258666126089601
Epoch: 400 Cost: 0.5468125736650521
Training Accuracy: 0.8714174709174652
Test Accuracy: 0.8350564227312657
Epoch: 500 Cost: 0.4930798476121502
Training Accuracy: 0.8802801833353979
Test Accuracy: 0.8420839245894993
Epoch: 600 Cost: 0.463209194527844
Training Accuracy: 0.8870144934064573
Test Accuracy: 0.8489762821812284
Epoch: 700 Cost: 0.442984539055873
Training Accuracy: 0.8922735616392077
Test Accuracy: 0.8520170281775795
Epoch: 800 Cost: 0.427826730770282
Training Accuracy: 0.8969470376918659
Test Accuracy: 0.854584769241165
Epoch: 900 Cost: 0.414929000276848

KeyboardInterrupt: 

In [11]:
np.save("W1_try1.npy", W1)
np.save("W2_try1.npy", W2)
np.save("W3_try1.npy", W3)

np.save("b1_try1.npy", b1)
np.save("b2_try1.npy", b2)
np.save("b3_try1.npy", b3)